In [1]:
from scipy import io
import numpy as np
dataslices_output='data_slices/output'
batch_size=1
num_harmonics=4 
audio_dir='../../assets/trainingdata/chords/'
sampleRate,audio=io.wavfile.read(audio_dir+'session_original.wav')
audio=audio.astype(np.float32)
_,audio_hi=io.wavfile.read(audio_dir+'session_eq high.wav')
audio_hi=audio_hi.astype(np.float32)
_,audio_mid=io.wavfile.read(audio_dir+'session_eq mid.wav')
audio_mid=audio_mid.astype(np.float32)
_,audio_dist=io.wavfile.read(audio_dir+'session_tape dist.wav')
audio_dist=audio_dist.astype(np.float32)

In [2]:
import numpy as np
import seaborn as sns
from scipy import signal
num_frets=13
num_strings=6
class Filter:
    def __init__(self, fret,stringid,harmonic,center_freq, bw,sample_rate):
        self.id=fret*num_strings*num_harmonics+stringid*num_harmonics+harmonic
        self.sample_rate=sample_rate
            
        # create the filter
        N = 2
        low = (center_freq-bw/2) 
        high = (center_freq+bw/2) 
        #self.b, self.a = signal.ellip(N,0.5,40,[low, high], btype='band',fs=sampleRate)
        self.b, self.a =signal.butter(N, [low, high], btype='band',fs=sampleRate)
        
    def process(self,input_audio,filterbank_out: np.array):
        f=np.abs(signal.filtfilt(self.b, self.a, input_audio))
        filterbank_out[self.id]=f


class HarmonicGroup:
    def __init__(self,fret,stringid ,center_freq, bw,sample_rate):
        self.harmonics=[]
        
        for h in range(1,num_harmonics+1):
            self.harmonics.append(Filter(fret,stringid,h-1,center_freq*h,bw,sample_rate))
    
            
    def process(self, input_audio, filterbank_out: np.array):
        res=filterbank_out
        for h in self.harmonics:
            #filterbank_out.append(h.process(input_audio))
            h.process(input_audio,filterbank_out)
            
        return res
    def get_num_filters(self):
        return len(self.harmonics)
          
            
class Fret:
    def __init__(self,fret,s0,s1,s2,s3,s4,s5, bw,sample_rate):
        
        self.strings=[]
        self.strings.append(HarmonicGroup(fret,0,s0,bw,sample_rate))
   
        self.strings.append(HarmonicGroup(fret,1,s1,bw,sample_rate))

        self.strings.append(HarmonicGroup(fret,2,s2,bw,sample_rate))

        self.strings.append(HarmonicGroup(fret,3,s3,bw,sample_rate))

        self.strings.append(HarmonicGroup(fret,4,s4,bw,sample_rate))

        self.strings.append(HarmonicGroup(fret,5,s5,bw,sample_rate))

        
    def process(self, input_audio, filterbank_out: np.array):
    
        for h in self.strings:
            #filterbank_out.append(h.process(input_audio,filterbank_out))
            h.process(input_audio,filterbank_out)
      
    def get_num_filters(self):
        res=0
        for h in self.strings:
            res=res+h.get_num_filters()
            
        return res
            
class FretBoard:
    def __init__(self,bw,sample_rate):
        self.frets=[]
       
        self.frets.append(Fret(0,82,11,147,196,247,329,bw,sample_rate))
        self.frets.append(Fret(1,87,117,156,208,262,349,bw,sample_rate))
        self.frets.append(Fret(2,92,123,165,220,277,370,bw,sample_rate))
        self.frets.append(Fret(3,98,131,175,233,294,392,bw,sample_rate))
        self.frets.append(Fret(4,104,139,185,247,311,415,bw,sample_rate))
        self.frets.append(Fret(5,110,147,196,262,329,440,bw,sample_rate))
        self.frets.append(Fret(6,117,156,208,277,349,466,bw,sample_rate))
        self.frets.append(Fret(7,123,165,220,294,370,494,bw,sample_rate))
        self.frets.append(Fret(8,131,175,233,311,392,523,bw,sample_rate))
        self.frets.append(Fret(9,139,185,247,329,415,554,bw,sample_rate))
        self.frets.append(Fret(10,147,196,262,349,440,587,bw,sample_rate))
        self.frets.append(Fret(11,156,208,277,370,466,622,bw,sample_rate))
        self.frets.append(Fret(12,165,220,294,392,494,659,bw,sample_rate))
        
    def process(self, input_audio, filterbank_out: np.array):
      
        for h in self.frets:
            # filterbank_out.append(h.process(input_audio,filterbank_out))    
            res=h.process(input_audio,filterbank_out)
       
    def get_num_filters(self):
        res=0
        for h in self.frets:
            res=res+h.get_num_filters()
            
        return res
    
    
filter =FretBoard(20,sampleRate)

filter_hi =FretBoard(20,sampleRate)
filter_mid =FretBoard(20,sampleRate)
filter_dist =FretBoard(20,sampleRate)
numfilters=filter.get_num_filters()
print('num filters:'+str(numfilters)+' audio samples '+str(len(audio)))


filterbank_out=np.zeros((numfilters,len(audio)),dtype=np.float32)
filterbank_out_hi=np.zeros((numfilters,len(audio)),dtype=np.float32)
filterbank_out_mid=np.zeros((numfilters,len(audio)),dtype=np.float32)
filterbank_out_dist=np.zeros((numfilters,len(audio)),dtype=np.float32)


filter.process(audio,filterbank_out)
filter_hi.process(audio_hi,filterbank_out_hi)
filter_mid.process(audio_mid,filterbank_out_mid)
filter_dist.process(audio_dist,filterbank_out_dist)

num filters:312 audio samples 12576000


In [3]:
from common import frame_size
def reshape_to_nn_input(indata):
    num_cols=indata.shape[1]
    num_rows=indata.shape[0]
    downsample_factor = frame_size
    # --- Downsampling the data ---
    print(f"reshape data by a factor of {downsample_factor}...")
    # Calculate the new number of columns after downsampling
    new_num_cols = num_cols // downsample_factor

    # Ensure the original number of columns is a multiple of the downsample_factor
    # If not, you might lose some data at the end or need a more complex aggregation.
    # For simplicity, we'll slice to a multiple of downsample_factor
    effective_cols = new_num_cols * downsample_factor
    data_sliced = indata[:, :effective_cols]
    print(data_sliced.shape)
    # Reshape the data for averaging:
    # -1: infer dimension
    # downsample_factor: group columns into blocks
    # num_rows: keep rows as isp
    # This reshapes (19, M*N) to (19, M, N)
    reshaped_data = data_sliced.reshape(num_rows, new_num_cols, downsample_factor,1)
    reshaped_data=np.swapaxes(reshaped_data,0,1)
    reshaped_data=np.swapaxes(reshaped_data,1,2)
    
    print('Reshaped the input data to  ')
    print(reshaped_data.shape)
    return reshaped_data
nn_input=reshape_to_nn_input(filterbank_out)
filterbank_out=None

nn_input_all=np.concatenate((nn_input,nn_input,nn_input,nn_input),axis=0)
nsamples=nn_input.shape[0];
nn_input_all[range(nsamples,2*nsamples)]=reshape_to_nn_input(filterbank_out_hi)
filterbank_out_hi=None

nn_input_all[range(2*nsamples,3*nsamples)]=reshape_to_nn_input(filterbank_out_mid)
filterbank_out_mid=None

nn_input_all[range(3*nsamples,4*nsamples)]=reshape_to_nn_input(filterbank_out_dist)
filterbank_out_dist=None


print(nn_input_all.shape)

reshape data by a factor of 256...
(312, 12576000)
Reshaped the input data to  
(49125, 256, 312, 1)
reshape data by a factor of 256...
(312, 12576000)
Reshaped the input data to  
(49125, 256, 312, 1)
reshape data by a factor of 256...
(312, 12576000)
Reshaped the input data to  
(49125, 256, 312, 1)
reshape data by a factor of 256...
(312, 12576000)
Reshaped the input data to  
(49125, 256, 312, 1)
(196500, 256, 312, 1)


In [4]:
from common import save_data_slices
output_dir_input = 'data_slices/input'

save_data_slices(output_dir_input,nn_input_all,batch_size)

Saving 196500 samples to disk
Saved slice 0/196500
Saved slice 1000/196500
Saved slice 2000/196500
Saved slice 3000/196500
Saved slice 4000/196500
Saved slice 5000/196500
Saved slice 6000/196500
Saved slice 7000/196500
Saved slice 8000/196500
Saved slice 9000/196500
Saved slice 10000/196500
Saved slice 11000/196500
Saved slice 12000/196500
Saved slice 13000/196500
Saved slice 14000/196500
Saved slice 15000/196500
Saved slice 16000/196500
Saved slice 17000/196500
Saved slice 18000/196500
Saved slice 19000/196500
Saved slice 20000/196500
Saved slice 21000/196500
Saved slice 22000/196500
Saved slice 23000/196500
Saved slice 24000/196500
Saved slice 25000/196500
Saved slice 26000/196500
Saved slice 27000/196500
Saved slice 28000/196500
Saved slice 29000/196500
Saved slice 30000/196500
Saved slice 31000/196500
Saved slice 32000/196500
Saved slice 33000/196500
Saved slice 34000/196500
Saved slice 35000/196500
Saved slice 36000/196500
Saved slice 37000/196500
Saved slice 38000/196500
Saved sl

In [5]:
nn_input_all=None
nn_input=None